# 🔍 FinSight IDX — Financial NLP Intelligence Platform

> **End-to-end demo**: PDF laporan tahunan IDX → AI summarization → Topic modeling → RAG Q&A → Financial analysis

[![Python](https://img.shields.io/badge/Python-3.10%2B-blue?logo=python)](https://www.python.org/)
[![Claude API](https://img.shields.io/badge/Claude-Anthropic-orange)](https://www.anthropic.com/)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Agathahah/finsight-idx/blob/main/notebooks/demo_finsight.ipynb)

**Author:** Agatha Silalahi — Data Scientist, Bank Indonesia Institute (BINS)  
**Stack:** Claude API · BERTopic · ChromaDB · FastMCP · sentence-transformers

---

## 📋 Demo Outline

| # | Module | Waktu | Token |
|---|--------|-------|-------|
| 1 | Setup & Environment | ~1 min | 0 |
| 2 | PDF Extraction — BBCA 2023 | ~2 min | 0 |
| 3 | Claude API Summarization | ~1 min | ~500 |
| 4 | Topic Modeling (Synthetic News) | ~3 min | 0 |
| 5 | RAG Q&A dengan Citations | ~1 min | ~300 |
| 6 | Financial Ratio Calculator | ~1 min | ~200 |
| 7 | Claude Tool Use — Analyst Agent | ~2 min | ~400 |
| 8 | Full Pipeline — Analisis BBCA | ~3 min | ~800 |


## 1️⃣ Setup & Environment

In [ ]:
# Install dependencies (Google Colab)
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone https://github.com/Agathahah/finsight-idx.git
    %cd finsight-idx
    !pip install -q anthropic python-dotenv loguru tenacity pdfplumber \
        sentence-transformers chromadb rouge-score pandas matplotlib \
        seaborn plotly fastmcp

print("✅ Dependencies ready")
print(f"   Running in: {'Google Colab' if IN_COLAB else 'Local environment'}")

In [ ]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
if not IN_COLAB:
    sys.path.insert(0, os.path.abspath('..'))
    from dotenv import load_dotenv
    load_dotenv('../.env')
else:
    # Set your API key in Colab Secrets or directly:
    # os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
    from google.colab import userdata
    try:
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        print("⚠️  Set ANTHROPIC_API_KEY in Colab Secrets (🔑 icon on left sidebar)")

# Verify API key
api_key = os.getenv('ANTHROPIC_API_KEY', '')
if api_key.startswith('sk-ant-'):
    print(f"✅ ANTHROPIC_API_KEY loaded ({api_key[:20]}...)")
else:
    print("❌ ANTHROPIC_API_KEY not set — API features will be skipped")

SKIP_API = not api_key.startswith('sk-ant-')

## 2️⃣ PDF Extraction — BBCA Annual Report 2023

Mengekstrak teks dari laporan tahunan PT Bank Central Asia Tbk (BBCA) 2023.
Dokumen ini tersedia publik di [idx.co.id](https://www.idx.co.id).


In [ ]:
from src.nlp.pdf_extractor import PDFExtractor
import pandas as pd

PDF_PATH = 'data/raw/bbca_laporan_tahunan_2023.pdf'

extractor = PDFExtractor()
doc = extractor.extract(PDF_PATH)

print(f"📄 Document  : {doc.metadata['filename']}")
print(f"📑 Pages     : {doc.total_pages:,}")
print(f"📂 Sections  : {len(doc.sections):,}")
print(f"📝 Characters: {len(doc.raw_text):,}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Section type distribution
from collections import Counter
type_counts = Counter(s.section_type for s in doc.sections)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')

COLORS = {
    'financial': '#00d4aa',
    'risk':      '#ff6b6b',
    'outlook':   '#ffd93d',
    'governance':'#74b9ff',
    'general':   '#a29bfe',
}

# Pie chart
labels = list(type_counts.keys())
sizes  = list(type_counts.values())
colors = [COLORS.get(l, '#888') for l in labels]

ax1 = axes[0]
ax1.set_facecolor('#0f1117')
wedges, texts, autotexts = ax1.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    textprops={'color': 'white', 'fontsize': 10},
    wedgeprops={'edgecolor': '#0f1117', 'linewidth': 2}
)
for at in autotexts:
    at.set_color('white')
    at.set_fontweight('bold')
ax1.set_title('Section Type Distribution', color='white', fontsize=13, pad=15)

# Bar chart — top 10 sections by char count
top_sections = sorted(doc.sections, key=lambda s: s.char_count, reverse=True)[:10]
ax2 = axes[1]
ax2.set_facecolor('#1a1d27')
bar_colors = [COLORS.get(s.section_type, '#888') for s in top_sections]
bars = ax2.barh(
    [s.title[:35] + ('…' if len(s.title) > 35 else '') for s in top_sections],
    [s.char_count for s in top_sections],
    color=bar_colors, edgecolor='none', height=0.6
)
ax2.set_xlabel('Character Count', color='#aaa', fontsize=10)
ax2.set_title('Top 10 Sections by Size', color='white', fontsize=13, pad=15)
ax2.tick_params(colors='white', labelsize=9)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['bottom'].set_color('#333')
ax2.spines['left'].set_color('#333')
ax2.set_facecolor('#1a1d27')
fig.patch.set_facecolor('#0f1117')

# Legend
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in COLORS.items()]
ax2.legend(handles=legend_patches, loc='lower right',
           facecolor='#2a2d37', edgecolor='none',
           labelcolor='white', fontsize=9)

plt.tight_layout(pad=2)
plt.savefig('section_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print(f"\n📊 Section breakdown: {dict(type_counts)}")

## 3️⃣ Claude API Summarization

Memanggil Claude API untuk meringkas teks laporan keuangan BBCA.
Menggunakan `FinancialTask` enum untuk berbagai jenis analisis.


In [ ]:
from src.api.client import analyze_financial_text, FinancialTask

# Sample text dari ikhtisar keuangan BBCA 2023
SAMPLE_TEXT = '''
BCA membukukan laba bersih sebesar Rp 48,6 triliun pada tahun 2023,
meningkat 19,5% dibandingkan tahun sebelumnya sebesar Rp 40,7 triliun.
Total aset BCA tumbuh 8,7% menjadi Rp 1.408 triliun.
Kredit yang disalurkan meningkat 13,8% menjadi Rp 793,2 triliun,
didorong oleh kredit korporasi yang tumbuh 16,2% dan kredit UKM 14,5%.
Rasio kecukupan modal (CAR) tercatat 25,9%, jauh di atas minimum regulator 14%.
Dana Pihak Ketiga (DPK) tumbuh 7,6% menjadi Rp 1.063 triliun.
BCA menargetkan pertumbuhan kredit 10-12% pada tahun 2024.
Risiko utama: tekanan suku bunga global dan pelemahan nilai tukar rupiah.
'''

if not SKIP_API:
    results = {}
    for task in [FinancialTask.SUMMARIZE, FinancialTask.KEY_METRICS,
                 FinancialTask.SENTIMENT, FinancialTask.RISK_FACTORS]:
        result = analyze_financial_text(SAMPLE_TEXT, task=task)
        results[task.value] = result
        print(f"\n{'='*50}")
        print(f"📌 {task.value.upper()}")
        print(f"{'='*50}")
        print(result['result'])
        print(f"[Tokens: {result['usage']['input_tokens']} in / {result['usage']['output_tokens']} out]")
else:
    print("⏭️  Skipping API call — ANTHROPIC_API_KEY not set")
    print("   Demo output:")
    print("   SUMMARIZE: BCA membukukan kinerja keuangan yang kuat...")
    print("   KEY_METRICS: {laba_bersih: 48.6T, ROE: 23.1%, CAR: 25.9%}")

In [ ]:
# Visualisasi token usage
if not SKIP_API and 'results' in dir():
    import matplotlib.pyplot as plt

    tasks = list(results.keys())
    input_tok  = [results[t]['usage']['input_tokens'] for t in tasks]
    output_tok = [results[t]['usage']['output_tokens'] for t in tasks]

    x = np.arange(len(tasks))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('#0f1117')
    ax.set_facecolor('#1a1d27')

    b1 = ax.bar(x - width/2, input_tok,  width, label='Input tokens',
                color='#00d4aa', alpha=0.85)
    b2 = ax.bar(x + width/2, output_tok, width, label='Output tokens',
                color='#ffd93d', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(tasks, color='white', fontsize=11)
    ax.set_ylabel('Token count', color='#aaa')
    ax.set_title('Token Usage per Task — Claude API', color='white', fontsize=13)
    ax.legend(facecolor='#2a2d37', edgecolor='none', labelcolor='white')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_color('#333')

    for bar in b1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(int(bar.get_height())), ha='center', va='bottom',
                color='white', fontsize=9)

    plt.tight_layout()
    plt.savefig('token_usage.png', dpi=150, bbox_inches='tight',
                facecolor='#0f1117')
    plt.show()

## 4️⃣ Topic Modeling — Berita Keuangan Indonesia

Menggunakan BERTopic + multilingual sentence-transformers untuk mengidentifikasi
topik dominan dari 500 artikel berita keuangan Indonesia.


In [ ]:
from src.nlp.news_scraper import generate_sample_dataset
import json, tempfile
from pathlib import Path

# Generate 500 artikel berita keuangan sintetis
with tempfile.TemporaryDirectory() as tmpdir:
    path = generate_sample_dataset(n_articles=500, output_dir=tmpdir)
    news_texts = [
        json.loads(line)['text']
        for line in path.read_text().strip().split('\n')
    ]

print(f"✅ Loaded {len(news_texts)} artikel berita keuangan")
print(f"\nContoh artikel:")
for i, text in enumerate(news_texts[:3]):
    print(f"  [{i+1}] {text[:120]}...")

In [ ]:
try:
    from src.nlp.topic_modeler import FinancialTopicModeler
    import tempfile

    print("🔄 Running BERTopic (pertama kali download model ~120MB)...")
    modeler = FinancialTopicModeler(min_topic_size=15)
    result  = modeler.fit(news_texts, model_name='bbca_news_2023')

    print(result.summary())

    # Save
    with tempfile.TemporaryDirectory() as tmpdir:
        modeler.save(result, 'topics_demo.json')
        modeler.save_csv(result, news_texts, 'topic_assignments_demo.csv')

    print(f"\n✅ Topic modeling selesai")
    print(f"   Topics      : {result.n_topics}")
    print(f"   Outliers    : {result.outlier_count} ({result.outlier_count/result.n_documents*100:.1f}%)")

except ImportError:
    print("⚠️  BERTopic tidak terinstall — jalankan: pip install bertopic")
    # Fallback data untuk visualisasi
    result = None

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Visualisasi topic distribution
if result and result.topics:
    top_topics = sorted(result.topics, key=lambda t: t.size, reverse=True)[:8]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.patch.set_facecolor('#0f1117')

    # Bar chart — topic sizes
    ax1 = axes[0]
    ax1.set_facecolor('#1a1d27')
    colors_grad = plt.cm.plasma(np.linspace(0.2, 0.9, len(top_topics)))
    bars = ax1.barh(
        [t.label[:30] for t in reversed(top_topics)],
        [t.size for t in reversed(top_topics)],
        color=list(reversed(colors_grad)), height=0.6
    )
    ax1.set_xlabel('Jumlah Artikel', color='#aaa')
    ax1.set_title('Top Topics — Berita Keuangan IDX', color='white', fontsize=12)
    ax1.tick_params(colors='white', labelsize=9)
    for spine in ax1.spines.values():
        spine.set_color('#333')

    for bar, topic in zip(bars, reversed(top_topics)):
        ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f"{bar.get_width()}", va='center', color='white', fontsize=9)

    # Word cloud per topic (top words)
    ax2 = axes[1]
    ax2.set_facecolor('#1a1d27')
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, len(top_topics))
    ax2.axis('off')
    ax2.set_title('Top Words per Topic', color='white', fontsize=12)

    for i, topic in enumerate(top_topics):
        y = len(top_topics) - i - 0.5
        ax2.text(0, y, f"#{topic.topic_id}", color='#ffd93d',
                 fontsize=9, fontweight='bold', va='center')
        words_str = ' · '.join(topic.representative_words[:5])
        ax2.text(1, y, words_str, color='#a0a0b0', fontsize=8, va='center')

    plt.tight_layout(pad=2)
    plt.savefig('topic_distribution.png', dpi=150, bbox_inches='tight',
                facecolor='#0f1117')
    plt.show()
else:
    # Fallback demo visualization
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('#0f1117')
    ax.set_facecolor('#1a1d27')
    demo_topics = ['Suku Bunga BI','Pasar Saham IHSG','Inflasi & Harga',
                   'Kredit Perbankan','Nilai Tukar Rupiah','Obligasi & SBN',
                   'Komoditas','Digital & Fintech']
    demo_sizes  = [87, 72, 65, 58, 51, 44, 38, 32]
    colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(demo_topics)))
    ax.barh(demo_topics[::-1], demo_sizes[::-1], color=colors[::-1], height=0.6)
    ax.set_title('Topic Distribution (Demo Data)', color='white', fontsize=13)
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_color('#333')
    plt.tight_layout()
    plt.savefig('topic_distribution.png', dpi=150, bbox_inches='tight',
                facecolor='#0f1117')
    plt.show()

## 5️⃣ RAG Q&A — Tanya Jawab Laporan BBCA

Hybrid retrieval (BM25 + semantic search) + Claude API untuk menjawab
pertanyaan tentang laporan tahunan dengan citations halaman.


In [ ]:
from src.rag.retriever import RAGRetriever

retriever = RAGRetriever(persist_dir='data/vectorstore')

QUESTIONS = [
    "Berapa laba bersih BCA tahun 2023?",
    "Apa risiko utama yang dihadapi BCA?",
    "Bagaimana target pertumbuhan kredit BCA 2024?",
]

print("🔍 Hybrid Retrieval Results (BM25 + Semantic + RRF)\n")
print("="*65)

retrieval_data = []
for q in QUESTIONS:
    chunks = retriever.retrieve(q, top_k=3, company_filter='BBCA')
    print(f"\n❓ {q}")
    for chunk in chunks:
        print(f"   📍 {chunk.citation()} | score={chunk.rrf_score:.4f}")
        print(f"      {chunk.text[:100]}...")
    retrieval_data.append({
        'question': q,
        'chunks': len(chunks),
        'top_page': chunks[0].page_number if chunks else 0,
        'top_score': chunks[0].rrf_score if chunks else 0,
    })

In [ ]:
if not SKIP_API:
    from src.rag.qa_chain import FinancialQAChain

    qa = FinancialQAChain(persist_dir='data/vectorstore')

    print("💬 RAG Q&A dengan Citations\n")
    print("="*65)

    qa_results = []
    for q in QUESTIONS[:2]:  # 2 pertanyaan untuk hemat token
        response = qa.ask(q, company_filter='BBCA', year_filter=2023)
        print(f"\n❓ {q}")
        print(f"💡 {response.answer[:400]}")
        print(f"\n📚 Sources ({len(response.sources)}):")
        for src in response.sources[:2]:
            print(f"   {src.citation} — hal. {src.page_number}")
        print(f"\n⚡ {response.input_tokens} in / {response.output_tokens} out tokens | {response.latency_ms:.0f}ms")
        print("-"*65)
        qa_results.append(response)
else:
    print("⏭️  Skipping RAG Q&A — ANTHROPIC_API_KEY not set")

## 6️⃣ Financial Ratio Calculator

Menghitung dan memvisualisasikan rasio keuangan fundamental
untuk saham-saham perbankan IDX.


In [ ]:
from src.api.tools import hitung_rasio_keuangan, bandingkan_emiten
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Data emiten perbankan IDX 2023
BANKING_DATA = {
    'BBCA': dict(emiten='BBCA', harga_saham=9500, eps=485,
                 book_value_per_share=2089, laba_bersih=48600,
                 ekuitas=210000, total_hutang=900000,
                 total_aset=1408000, pendapatan=149200),
    'BBRI': dict(emiten='BBRI', harga_saham=5500, eps=320,
                 book_value_per_share=1850, laba_bersih=60400,
                 ekuitas=332000, total_hutang=1200000,
                 total_aset=1900000, pendapatan=213000),
    'BMRI': dict(emiten='BMRI', harga_saham=6200, eps=410,
                 book_value_per_share=2100, laba_bersih=55100,
                 ekuitas=280000, total_hutang=1100000,
                 total_aset=2174000, pendapatan=178000),
    'BBNI': dict(emiten='BBNI', harga_saham=5400, eps=280,
                 book_value_per_share=1650, laba_bersih=20900,
                 ekuitas=155000, total_hutang=850000,
                 total_aset=1050000, pendapatan=98000),
}

# Hitung rasio untuk semua emiten
all_ratios = {}
for ticker, data in BANKING_DATA.items():
    result = hitung_rasio_keuangan(**data)
    all_ratios[ticker] = result['rasio']
    print(f"\n{ticker}:")
    for k, v in result['rasio'].items():
        print(f"  {k:5s}: {v}")

print("\n✅ Rasio keuangan berhasil dihitung")

In [ ]:
# Visualisasi perbandingan rasio
import matplotlib.pyplot as plt
import numpy as np

tickers = list(all_ratios.keys())
metrics_to_plot = ['PER', 'PBV']

# Extract numeric values
def extract_num(val):
    if val is None: return 0
    s = str(val).replace('%','').replace('x','')
    try: return float(s)
    except: return 0

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('Perbandingan Rasio Keuangan — Perbankan IDX 2023',
             color='white', fontsize=14, fontweight='bold', y=1.01)

metric_configs = [
    ('PER',  'Price-to-Earnings (x)', '#00d4aa', 'lower better'),
    ('PBV',  'Price-to-Book Value (x)', '#74b9ff', 'lower better'),
    ('ROE',  'Return on Equity (%)', '#ffd93d', 'higher better'),
    ('DER',  'Debt-to-Equity (x)', '#ff6b6b', 'lower better'),
    ('NPM',  'Net Profit Margin (%)', '#a29bfe', 'higher better'),
    ('ROA',  'Return on Assets (%)', '#fd79a8', 'higher better'),
]

for ax, (metric, title, color, note) in zip(axes.flat, metric_configs):
    ax.set_facecolor('#1a1d27')
    values = [extract_num(all_ratios[t].get(metric)) for t in tickers]
    bars   = ax.bar(tickers, values, color=color, alpha=0.85,
                    edgecolor='none', width=0.5)

    # Highlight best
    best_idx = values.index(min(values) if 'lower' in note else max(values))
    bars[best_idx].set_edgecolor('white')
    bars[best_idx].set_linewidth(2)

    ax.set_title(f'{title}\n({note})', color='white', fontsize=10)
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_color('#333')

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(values)*0.02,
                f'{val:.1f}', ha='center', color='white', fontsize=9)

plt.tight_layout(pad=2)
plt.savefig('ratio_comparison.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print("\n⭐ Bar dengan border putih = emiten terbaik pada metrik tersebut")

## 7️⃣ Claude Tool Use — Financial Analyst Agent

Multi-turn conversation dengan Claude yang secara otomatis memanggil tools
untuk menjawab pertanyaan keuangan kompleks.


In [ ]:
if not SKIP_API:
    from src.api.tools import FinancialAnalystAgent

    agent = FinancialAnalystAgent()

    print("🤖 Financial Analyst Agent — Multi-turn Conversation")
    print("="*65)

    # Turn 1: hitung rasio
    r1 = agent.chat(
        "Hitung PER dan ROE untuk BBCA dengan data: "
        "harga saham Rp 9.500, EPS Rp 485, "
        "laba bersih Rp 48.600 miliar, ekuitas Rp 210.000 miliar"
    )
    print(f"\n👤 User: Hitung PER dan ROE BBCA...")
    print(f"🔧 Tools: {r1.tools_used}")
    print(f"🤖 Agent:\n{r1.answer}")
    print(f"\n⚡ {r1.input_tokens} in / {r1.output_tokens} out tokens | {r1.turns} turns")

    # Turn 2: bandingkan (lanjutkan history)
    print("\n" + "="*65)
    r2 = agent.chat(
        "Bandingkan dengan BBRI: ROE 18.2%, DER 5.1x, EPS Rp 320, "
        "harga Rp 5.500, pertumbuhan laba 17.5%",
        history=r1.conversation_history
    )
    print(f"\n👤 User: Bandingkan dengan BBRI...")
    print(f"🔧 Tools: {r2.tools_used}")
    print(f"🤖 Agent:\n{r2.answer}")
    print(f"\n⚡ Total tokens: {r1.input_tokens + r2.input_tokens} in | {r1.turns + r2.turns} turns")
else:
    print("⏭️  Skipping — ANTHROPIC_API_KEY not set")

## 8️⃣ Full Pipeline — Analisis BBCA End-to-End

Menjalankan `FinSightOrchestrator` — pipeline lengkap dari PDF ke laporan Markdown.

Pipeline: PDF → Summarize → Sentiment → RAG → Generate Report


In [ ]:
if not SKIP_API:
    from src.api.agent import FinSightOrchestrator
    import time

    print("🚀 FinSight IDX — Full Analysis Pipeline")
    print("="*65)
    print(f"   Emiten : BBCA")
    print(f"   Tahun  : 2023")
    print(f"   PDF    : bbca_laporan_tahunan_2023.pdf (747 hal)")
    print("\n⏳ Running pipeline...\n")

    start = time.time()
    orchestrator = FinSightOrchestrator(output_dir='data/processed')

    report = orchestrator.analyze(
        emiten='BBCA',
        tahun=2023,
        pdf_path='data/raw/bbca_laporan_tahunan_2023.pdf',
        skip_topic_modeling=True,  # Hemat waktu di demo
        skip_rag=False,
    )

    elapsed = time.time() - start

    print(f"✅ Pipeline selesai!")
    print(f"   Steps     : {report.steps_completed}")
    print(f"   Duration  : {elapsed:.1f}s")
    print(f"   Tokens in : {report.total_input_tokens:,}")
    print(f"   Tokens out: {report.total_output_tokens:,}")
    print(f"\n{'='*65}")
    print("📄 MARKDOWN REPORT (preview 2000 chars):")
    print("="*65)
    print(report.markdown_report[:2000])
    print("\n... [truncated] ...")

    # Save report
    saved_path = report.save('data/processed')
    print(f"\n💾 Full report saved → {saved_path}")
else:
    print("⏭️  Skipping — ANTHROPIC_API_KEY not set")
    print("\nDemo output preview:")
    print("""
# Laporan Analisis FinSight IDX
## BBCA — 2023

## Ikhtisar Perusahaan
PT Bank Central Asia Tbk (BBCA) adalah bank swasta terbesar di Indonesia...

## Kinerja Keuangan
- Laba bersih: Rp 48,6 triliun (+19,5% YoY)
- Total aset: Rp 1.408 triliun (+8,7%)
- ROE: 23,1% — excellent

## Kesimpulan
BBCA menunjukkan kinerja yang sangat solid di 2023...
    """)

## 9️⃣ Evaluation — ROUGE + Claude-as-Judge

Mengukur kualitas summarization secara otomatis (ROUGE) dan
menggunakan Claude sebagai judge (skala 1-5).


In [ ]:
from src.api.evaluator import SummaryEvaluator, get_builtin_eval_dataset
import pandas as pd

evaluator = SummaryEvaluator(output_dir='data/processed')
dataset   = get_builtin_eval_dataset()

print(f"📊 Evaluation Dataset: {len(dataset)} samples")
print(f"   Companies: {list(set(s.company for s in dataset))}")

# ROUGE only (no API cost)
report_rouge = evaluator.evaluate_dataset(dataset, skip_judge=True)

print(f"\n📈 ROUGE Scores (Automatic Metrics):")
print(f"   ROUGE-1 F1: {report_rouge.avg_rouge1_f1:.3f}")
print(f"   ROUGE-2 F1: {report_rouge.avg_rouge2_f1:.3f}")
print(f"   ROUGE-L F1: {report_rouge.avg_rougeL_f1:.3f}")

# Visualisasi ROUGE per sample
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')

# ROUGE scores per sample
samples  = [r.sample_id.replace('_', '\n') for r in report_rouge.results]
r1_scores = [r.rouge.rouge1_f1 for r in report_rouge.results]
rl_scores = [r.rouge.rougeL_f1 for r in report_rouge.results]

ax1 = axes[0]
ax1.set_facecolor('#1a1d27')
x = np.arange(len(samples))
ax1.bar(x - 0.2, r1_scores, 0.35, label='ROUGE-1', color='#00d4aa', alpha=0.85)
ax1.bar(x + 0.2, rl_scores, 0.35, label='ROUGE-L', color='#74b9ff', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(samples, rotation=45, ha='right', fontsize=7, color='white')
ax1.set_ylabel('F1 Score', color='#aaa')
ax1.set_title('ROUGE Scores per Sample', color='white', fontsize=12)
ax1.legend(facecolor='#2a2d37', edgecolor='none', labelcolor='white')
ax1.tick_params(colors='white')
for spine in ax1.spines.values(): spine.set_color('#333')

# Avg metrics summary
ax2 = axes[1]
ax2.set_facecolor('#1a1d27')
metrics = ['ROUGE-1\nF1', 'ROUGE-2\nF1', 'ROUGE-L\nF1']
values  = [report_rouge.avg_rouge1_f1, report_rouge.avg_rouge2_f1, report_rouge.avg_rougeL_f1]
colors_m = ['#00d4aa', '#ffd93d', '#74b9ff']
bars = ax2.bar(metrics, values, color=colors_m, alpha=0.85, width=0.4)
ax2.set_ylim(0, 1.0)
ax2.axhline(0.5, color='white', linestyle='--', alpha=0.3, label='0.5 baseline')
ax2.set_title('Average ROUGE Scores\n(FinSight IDX Eval Set)', color='white', fontsize=12)
ax2.tick_params(colors='white')
for spine in ax2.spines.values(): spine.set_color('#333')
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', color='white', fontsize=12, fontweight='bold')

plt.tight_layout(pad=2)
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

print("\n💡 Note: ROUGE rendah untuk summarization adalah normal —")
print("   model melakukan parafrase, bukan copy-paste.")
print("   Claude-as-Judge score: 4.40/5.00 (dari eval sebelumnya)")

## 🏁 Summary — FinSight IDX Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    FinSight IDX Pipeline                    │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  PDF (747 hal)  ──→  PDFExtractor  ──→  423 sections       │
│                           │                                 │
│                     DocumentSummarizer                      │
│                           │                                 │
│                    Claude API (Sonnet)                      │
│                    ┌──────┴──────┐                          │
│              Summarize      Key Metrics                     │
│              Risk Factors   Outlook                         │
│                    │                                        │
│              ┌─────┼─────────────────┐                     │
│              │     │                 │                     │
│           ChromaDB  BERTopic    Sentiment                   │
│           (RAG)    (Topics)    (News vs Report)             │
│              │                                              │
│         FinancialQAChain  ←──  RAGRetriever                 │
│         (Claude + Citations)   (BM25 + Semantic)            │
│                    │                                        │
│          FinancialAnalystAgent (Tool Use)                   │
│                    │                                        │
│              MCP Server (FastMCP)                           │
│         ┌──────────┤                                        │
│    Claude Desktop  MCP Inspector                            │
└─────────────────────────────────────────────────────────────┘
```

### 📊 Metrics Summary

| Component | Metric | Value |
|-----------|--------|-------|
| PDF Extraction | Pages extracted | 747 |
| PDF Extraction | Sections detected | 423 |
| Summarization | ROUGE-L F1 | ~0.16 |
| Evaluation | Claude Judge score | 4.40/5.00 |
| RAG | Chunks indexed | 134,764 |
| Test Suite | Unit tests passing | 143/143 |

### 🔗 Links
- **GitHub**: [github.com/Agathahah/finsight-idx](https://github.com/Agathahah/finsight-idx)
- **LinkedIn**: [Agatha Silalahi](https://www.linkedin.com/in/agatha-silalahi-722507215/)
